![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/embeddings/ModernBertEmbeddings.ipynb)

# ModernBERT Embeddings with Spark NLP

ModernBERT is a modernized bidirectional encoder that brings modern architectural improvements to the classic BERT model. It incorporates **Flash Attention**, **unpadding**, and **GeGLU activation functions**, and supports sequence lengths up to **8192 tokens**.

## 0. Colab Setup

In [ ]:
# Only run this cell if you are on Google Colab
! wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [ ]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
import pyspark.sql.functions as F

# Use gpu=True to Start Spark with GPU support.
spark = sparknlp.start()

print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)

## 1. Basic Token Embeddings Pipeline

Extract token-level embeddings from text using the pretrained `modernbert-base` model.

**Pipeline stages:**
- `DocumentAssembler` converts raw text into Spark NLP `DOCUMENT` annotations
- `Tokenizer` splits documents into `TOKEN` annotations
- `ModernBertEmbeddings` generates 768-dimensional `WORD_EMBEDDINGS` for each token
- `EmbeddingsFinisher` converts Spark NLP embeddings into standard Spark ML vectors

In [ ]:
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

embeddings = ModernBertEmbeddings.pretrained("modernbert-base") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("embeddings")

embeddings_finisher = EmbeddingsFinisher() \
    .setInputCols(["embeddings"]) \
    .setOutputCols("finished_embeddings") \
    .setOutputAsVector(True) \
    .setCleanAnnotations(False)

pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    embeddings,
    embeddings_finisher
])

In [ ]:
sample_data = spark.createDataFrame([
    ["ModernBERT is a state-of-the-art encoder model."],
    ["Spark NLP makes it easy to use transformer models at scale."],
    ["The quick brown fox jumps over the lazy dog."]
]).toDF("text")

model = pipeline.fit(sample_data)
result = model.transform(sample_data)

In [ ]:
# Show the tokens and their corresponding embeddings
result.select("token.result", "finished_embeddings").show(truncate=80)

+-------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                   result|                                                             finished_embeddings|
+-------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                 [ModernBERT, is, a, state-of-the-art, encoder, model, .]|[[1.4624786376953125,-0.9653674364089966,0.7978458404541016,0.136210799217224...|
|[Spark, NLP, makes, it, easy, to, use, transformer, models, at, scale, .]|[[1.4134446382522583,-1.0480096340179443,0.4999319016933441,-0.53463619947433...|
|                 [The, quick, brown, fox, jumps, over, the, lazy, dog, .]|[[0.11401227861642838,0.06569014489650726,-0.19090411067008972,1.086695551872...|
+---------------------------------------------------------

## 2. Exploring the Embeddings

Let's inspect the embedding dimensions and look at the raw annotation metadata to understand what `ModernBertEmbeddings` produces.

In [ ]:
embeddings.getDimension()

768

In [ ]:
# Check embedding dimension — should be 768 for modernbert-base
import numpy as np

pdf = result.select("finished_embeddings").toPandas()
first_sentence_embeddings = pdf["finished_embeddings"][0]

print(f"Number of tokens in first sentence: {len(first_sentence_embeddings)}")
print(f"Embedding dimension per token: {len(first_sentence_embeddings[0])}")
print(f"First 10 values of the first token embedding: {first_sentence_embeddings[0][:10]}")

Number of tokens in first sentence: 7
Embedding dimension per token: 768
First 10 values of the first token embedding: [ 1.46247864 -0.96536744  0.79784584  0.1362108  -1.34380436 -0.52033955
 -0.04105116 -0.73480785  0.32259056  0.91760272]


In [ ]:
# Look at the full annotation structure (metadata, token text, etc.)
result.select(F.explode("embeddings").alias("emb")) \
    .select(
        F.col("emb.result").alias("token"),
        F.col("emb.begin").alias("begin"),
        F.col("emb.end").alias("end"),
        F.col("emb.metadata").alias("metadata")
    ).filter(F.col("token") != "") \
    .show(20, truncate=False)

+----------------+-----+---+------------------------------------------------------------------------------------------------+
|token           |begin|end|metadata                                                                                        |
+----------------+-----+---+------------------------------------------------------------------------------------------------+
|modernbert      |0    |9  |{isOOV -> false, pieceId -> 26944, isWordStart -> true, token -> modernbert, sentence -> 0}     |
|is              |11   |12 |{isOOV -> false, pieceId -> 310, isWordStart -> true, token -> is, sentence -> 0}               |
|a               |14   |14 |{isOOV -> false, pieceId -> 247, isWordStart -> true, token -> a, sentence -> 0}                |
|state-of-the-art|16   |31 |{isOOV -> false, pieceId -> 1375, isWordStart -> true, token -> state-of-the-art, sentence -> 0}|
|encoder         |33   |39 |{isOOV -> false, pieceId -> 32049, isWordStart -> true, token -> encoder, sentence -> 0}  

## 3. Token Similarity with Cosine Distance

Since each token gets a dense 768-dimensional vector, we can compute **cosine similarity** between tokens to see how semantically close they are according to ModernBERT.

In [ ]:
from numpy.linalg import norm

# Get embeddings for a sentence
similarity_data = spark.createDataFrame([
    ["The king sat on the throne while the queen watched from afar."]
]).toDF("text")

sim_result = model.transform(similarity_data)

# Extract tokens and embeddings
pdf_sim = sim_result.select("token.result", "finished_embeddings").toPandas()
tokens = pdf_sim["result"][0]
embeds = pdf_sim["finished_embeddings"][0]

print(f"Tokens: {tokens}\n")

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (norm(a) * norm(b))

# Compare selected token pairs
pairs = [
    ("king", "queen"),
    ("king", "throne"),
    ("king", "watched"),
    ("sat", "watched"),
]

for t1, t2 in pairs:
    if t1 in tokens and t2 in tokens:
        i1, i2 = tokens.index(t1), tokens.index(t2)
        sim = cosine_similarity(embeds[i1], embeds[i2])
        print(f"  cosine_similarity('{t1}', '{t2}') = {sim:.4f}")


Tokens: ['The', 'king', 'sat', 'on', 'the', 'throne', 'while', 'the', 'queen', 'watched', 'from', 'afar', '.']

  cosine_similarity('king', 'queen') = 0.8753
  cosine_similarity('king', 'throne') = 0.7414
  cosine_similarity('king', 'watched') = 0.6497
  cosine_similarity('sat', 'watched') = 0.8441


## 4. Parameter Tuning

`ModernBertEmbeddings` supports several configurable parameters:

| Parameter | Default | Description |
|---|---|---|
| `maxSentenceLength` | 8192 | Maximum token length per sentence (max 8192) |
| `batchSize` | 8 | Number of sentences processed together |
| `dimension` | 768 | Embedding vector size (set by the model) |
| `caseSensitive` | False | Whether tokenization is case-sensitive |

In [ ]:
# Example: Configure for faster inference with shorter sequences
embeddings_tuned = ModernBertEmbeddings.pretrained("modernbert-base") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("embeddings") \
    .setMaxSentenceLength(512) \
    .setBatchSize(16)

print(f"maxSentenceLength: {embeddings_tuned.getMaxSentenceLength()}")
print(f"batchSize:         {embeddings_tuned.getBatchSize()}")
print(f"dimension:         {embeddings_tuned.getDimension()}")
print(f"caseSensitive:     {embeddings_tuned.getCaseSensitive()}")

maxSentenceLength: 512
batchSize:         16
dimension:         768
caseSensitive:     False


In [ ]:
# maxSentenceLength must be between 1 and 8192
try:
    embeddings_tuned.setMaxSentenceLength(9000)
except ValueError as e:
    print(f"Expected error: {e}")

Expected error: ModernBERT models do not support sequences longer than 8192 because of trainable positional embeddings.


## 5. Saving and Loading the Pipeline

Fitted pipeline models (including ModernBERT embeddings) can be saved to disk and reloaded without re-downloading.

In [ ]:
# Save the fitted pipeline model
model.write().overwrite().save("./modernbert_pipeline")

# Reload it
from pyspark.ml import PipelineModel

loaded_model = PipelineModel.load("./modernbert_pipeline")

# Verify it works
verify_data = spark.createDataFrame([["Loading a saved model works!"]]).toDF("text")
verify_result = loaded_model.transform(verify_data)
verify_result.select("token.result", F.size("finished_embeddings").alias("num_embeddings")).show(truncate=False)

+------------------------------------+--------------+
|result                              |num_embeddings|
+------------------------------------+--------------+
|[Loading, a, saved, model, works, !]|6             |
+------------------------------------+--------------+



## What if you want to use a different ModernBERT Model?

Spark NLP's `ModernBertEmbeddings` works with **any ModernBERT-architecture model** from HuggingFace.

To use a different model, you need to:
1. **Export** the HuggingFace model to ONNX format using the `optimum` library
2. **Import** the ONNX model into Spark NLP with `ModernBertEmbeddings.loadSavedModel()`
3. **Save** it locally so it can be reloaded without re-exporting

> **Why ONNX?** Spark NLP runs on the JVM (Scala/Java), so it cannot directly load Python-based HuggingFace models. The ONNX format provides a cross-platform, optimized runtime that works within Spark NLP's JVM-based architecture.

### Step 1: Install the export dependencies

In [ ]:
!pip install -q optimum[onnxruntime] transformers

### Step 2: Export the model to ONNX

We use `optimum`'s `ORTModelForFeatureExtraction` to download the model from HuggingFace and export it to ONNX format in a single step. The tokenizer files are saved into an `assets/` subdirectory. Spark NLP reads `tokenizer.json` from there to reconstruct the BPE tokenizer on the JVM side.

> **Tip:** Change `model_id` below to any ModernBERT-compatible model on HuggingFace (e.g. `"answerdotai/ModernBERT-large"`).

The directory structure expected by `loadSavedModel` is:
```
onnx/
├── model.onnx          # The exported ONNX model
└── assets/
    └── tokenizer.json  # HuggingFace tokenizer (vocab + merges + special tokens)
```

In [ ]:
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer

model_id = "answerdotai/ModernBERT-base"
output_dir = "onnx"
assets_dir = "onnx/assets"

model = ORTModelForFeatureExtraction.from_pretrained(
    model_id,
    export=True,
    trust_remote_code=True
)
model.save_pretrained(output_dir)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.save_pretrained(assets_dir)

### Step 3: Import into Spark NLP and save

`loadSavedModel()` reads the ONNX model and the tokenizer files from the export directory. After importing, we **save** it as a Spark NLP model so that future pipelines can load it instantly with `ModernBertEmbeddings.load()`.

In [ ]:
imported_model = (
    ModernBertEmbeddings.loadSavedModel("onnx", spark)
        .setInputCols(["document", "token"])
        .setOutputCol("embeddings")
        .write().overwrite().save("modernbert-base")
)


### Step 4: Use the imported model in a pipeline

Now the model is saved locally and can be loaded just like any pretrained Spark NLP model. This is identical to how the built-in `modernbert-base` model works, the only difference is we're loading from a local path instead of the Spark NLP model hub.

In [ ]:
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

embeddings = ModernBertEmbeddings.load("modernbert-base") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("embeddings")

embeddings_finisher = EmbeddingsFinisher() \
    .setInputCols(["embeddings"]) \
    .setOutputCols("finished_embeddings") \
    .setOutputAsVector(True) \
    .setCleanAnnotations(False)

pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    embeddings,
    embeddings_finisher
])

sample_data = spark.createDataFrame([
    ["ModernBERT is a state-of-the-art encoder model."],
    ["Spark NLP makes it easy to use transformer models at scale."],
    ["The quick brown fox jumps over the lazy dog."]
]).toDF("text")

model = pipeline.fit(sample_data)
result = model.transform(sample_data)

result.select("token.result", "finished_embeddings").show(truncate=80)


+-------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                   result|                                                             finished_embeddings|
+-------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                 [ModernBERT, is, a, state-of-the-art, encoder, model, .]|[[1.4624786376953125,-0.9653674364089966,0.7978458404541016,0.136210799217224...|
|[Spark, NLP, makes, it, easy, to, use, transformer, models, at, scale, .]|[[1.4134446382522583,-1.0480096340179443,0.4999319016933441,-0.53463619947433...|
|                 [The, quick, brown, fox, jumps, over, the, lazy, dog, .]|[[0.11401227861642838,0.06569014489650726,-0.19090411067008972,1.086695551872...|
+---------------------------------------------------------

For more information, see:
- [Spark NLP ModernBertEmbeddings documentation](https://sparknlp.org/docs/en/transformers#modernbertembeddings)
- [Paper: Smarter, Better, Faster, Longer](https://arxiv.org/abs/2412.13663)
- [HuggingFace: answerdotai/ModernBERT-base](https://huggingface.co/answerdotai/ModernBERT-base)